# Lab 1 — Perception Audit and Five Encodings
SDA-DSC-112 · Tayseer

**Task:** Rank Saudi regions by latest-month digital adoption accurately and identify regions requiring intervention against the 65% target.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

TARGET=65.0
p=Path('../data/tayseer_services.csv')
if not p.exists(): p=Path('../data/dashboard_summary.csv')
df=pd.read_csv(p, parse_dates=['month'])
if 'scope' in df.columns:
    latest=df[df.scope=='region_latest'][['region','digital_adoption_pct','unique_users']].copy()
else:
    m=df.month.max(); x=df[df.month==m].copy()
    latest=(x.assign(weighted=x.digital_adoption_pct*x.unique_users).groupby('region',as_index=False).agg(weighted=('weighted','sum'),unique_users=('unique_users','sum')))
    latest['digital_adoption_pct']=latest.weighted/latest.unique_users
latest=latest.sort_values('digital_adoption_pct')
latest

## Five encodings
The same series is encoded five ways so decoding accuracy can be compared for the ranking task.

In [ ]:
s=latest.set_index('region')['digital_adoption_pct']
fig,ax=plt.subplots(figsize=(8,5)); s.plot.barh(ax=ax); ax.set_xlim(0,100); ax.set_title('1. Sorted bar — ranking is immediate'); plt.show()
fig,ax=plt.subplots(figsize=(8,5)); latest.sample(frac=1,random_state=1).plot.bar(x='region',y='digital_adoption_pct',ax=ax,legend=False); ax.set_ylim(0,100); ax.set_title('2. Unsorted bar'); plt.xticks(rotation=70); plt.show()
fig,ax=plt.subplots(figsize=(7,7)); ax.pie(s.values,labels=s.index); ax.set_title('3. Pie — angle/area makes ranking difficult'); plt.show()
fig,ax=plt.subplots(figsize=(9,3)); ax.scatter(range(len(s)),np.ones(len(s)),s=(s.values**2)/3,alpha=.6); ax.set_xticks(range(len(s)),s.index,rotation=60); ax.set_title('4. Bubble/area'); ax.set_yticks([]); plt.show()
fig,ax=plt.subplots(figsize=(10,2)); im=ax.imshow([s.values],aspect='auto'); ax.set_xticks(range(len(s)),s.index,rotation=60); ax.set_yticks([]); ax.set_title('5. Single-row heatmap — useful for pattern, weak for exact ranking'); plt.colorbar(im,ax=ax); plt.show()

## Ranking by decoding accuracy
1. **Sorted bar** — position/length on a common scale; best match for ranking.
2. **Unsorted bar** — accurate length, but ordering forces extra search.
3. **Pie** — angle is less accurately decoded than aligned position/length.
4. **Bubble/area** — area is difficult to compare precisely.
5. **Heatmap** — color intensity is effective for pattern detection but not exact ranking.

Cleveland–McGill principle used: position on a common scale > length > angle/area > color intensity for precise quantitative comparison.

## Bad-dashboard audit
The supplied audit target should be checked for: rainbow color without meaning, pie/angle encoding for ranking, 3-D or decorative ink, truncated bar baselines, dual axes that imply a relationship, legends that force eye travel, missing target/reference line, and insufficient emphasis on the actual lagging regions. These channels spend attention without improving the decision.

Under-encoded signals: the 65% target, ordered regional rank, the gap to target, and the few regions needing immediate intervention.

In [ ]:
# Corrected chart: exactly one semantic highlight — below target.
colors=['#D55E00' if v<TARGET else '#B8B8B8' for v in latest.digital_adoption_pct]
fig,ax=plt.subplots(figsize=(8,6))
ax.barh(latest.region,latest.digital_adoption_pct,color=colors)
ax.axvline(TARGET,color='black',linestyle='--',linewidth=1)
ax.set_xlim(0,100); ax.set_xlabel('Digital adoption (%)')
ax.set_title('Nine regions remain below the 65% target — latest month',loc='left')
for i,v in enumerate(latest.digital_adoption_pct): ax.text(v+0.7,i,f'{v:.1f}%',va='center',fontsize=8)
ax.spines[['top','right']].set_visible(False); ax.grid(False); plt.tight_layout(); plt.show()

### Conclusion
The sorted horizontal bar is the decision-ready encoding. It preserves an honest zero baseline, exposes rank immediately, adds the 65% reference, and uses one meaningful highlight for below-target regions.